In [20]:
import casadi as cs
import numpy as np
from time import perf_counter
from matplotlib import pyplot as plt
import pickle
import ctypes
import os

import problem_definition
from solver_banks import Solver 
from OCP_layout import Layout
from dynamics import Hopper 
import LUT_helper as helper

In [21]:
def initialize_mode(LUT, count, E0, phi_end=1.0):
    return np.column_stack([
        np.asarray(LUT([E0, phi])[:-1]).squeeze()
        for phi in np.linspace(0, phi_end, count)
    ])

In [22]:
def qp_data(w,p,linearize,con_min,con_max,w_min,w_max,delta):
    grad,H,A,residual = linearize(w,p)
    nominal = np.asarray(w).squeeze()
    return {
        "h": H, "g": grad, "a": A,
        "lbx": np.maximum(w_min-nominal, -delta),
        "ubx": np.minimum(w_max-nominal, delta),
        "lba": con_min-residual, "uba": con_max-residual,
    }

In [23]:
def evaluate(w,p,values,r_min,r_max,mu=1e3):
    J,r = values(w,p)
    violation = np.maximum(r_min-r,0) + np.maximum(r-r_max,0)
    merit = J + mu * np.sum(violation)
    return merit

In [24]:
def line_search(w_bar,p_bar,delta_w,values,g_min,g_max,mu=1e3):
    merit = evaluate(w_bar,p_bar,values,g_min,g_max,mu)
    alpha = 1.0
    while alpha > 1e-6:
        trial = w_bar + alpha*delta_w 
        if evaluate(trial,p_bar,values,g_min,g_max,mu) < merit:
            return False, alpha
        alpha *= 0.5
    return True, alpha

In [25]:
libc = ctypes.CDLL(None)
def solve_qp(qp,solver):
    # wrapper to remove unwanted output from hpipm
    stdout, null = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    try:  # CasADi 3.7.2's HPIPM plugin prints directly to C stdout
        os.dup2(null, 1)
        return solver(**qp)
    finally:
        libc.fflush(None)
        os.dup2(stdout, 1)
        os.close(stdout)
        os.close(null)

In [26]:
data = pickle.load(open('../data/dataset_2/dataset_21.pkl','rb'))
K = data["K"]
W = data["W"]
Nf = data["Nf"]
Ns = data["Ns"]
locomotion_NMM = np.array(data["locomotion"][2])
switching_NMM = np.array(data["switch"][2])
Emin = max(min(locomotion_NMM[:,-1]),min(switching_NMM[:,-1]))
Emax = min(max(locomotion_NMM[:,-1]),max(switching_NMM[:,-1]))

dynamics = Hopper(K,W,Nf=15,Ns=20)


In [27]:
E_grid = np.linspace(Emin, Emax, 20)
locomotion_NMM_sampled = helper.energy_sample(locomotion_NMM,E_grid)
switching_NMM_sampled = helper.energy_sample(switching_NMM,E_grid)

phi_grid = np.linspace(0,1,20)
ss_loc,sf_loc = helper.surface(locomotion_NMM_sampled,phi_grid,dynamics)
ss_swi,sf_swi = helper.surface(switching_NMM_sampled,phi_grid,dynamics)

values_sloc = ss_loc.ravel(order="C")
values_floc = sf_loc.ravel(order="C")
values_sswi = ss_swi.ravel(order="C")
values_fswi = sf_swi.ravel(order="C")
LUT_sloc = cs.interpolant('LUT_sloc', 'bspline',[E_grid,phi_grid],values_sloc)
LUT_floc = cs.interpolant('LUT_floc', 'bspline',[E_grid,phi_grid],values_floc)
LUT_sswi = cs.interpolant('LUT_sswi', 'bspline',[E_grid,phi_grid],values_sswi)
LUT_fswi = cs.interpolant('LUT_fswi', 'bspline',[E_grid,phi_grid],values_fswi)

In [28]:
E0 = 4
E_inf = max(Emin,E0-0.2)
E_sup = min(Emax,E0+0.2)

x_bounds = {
    "f" : {1: (0.0, np.inf)}, 
    "s" : {}
}
u_bounds = {
    "f" : (-1,1), 
    "s" : (-1,1)
}
t_bounds = {
    "f": (1e-2, 10.0),
    "s": (1e-2, 10.0),
}
E_bounds = (E_inf,E_sup)
phi_bounds = (0.01,0.99)

bounds = (x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds)

In [29]:
modes = ("s", "f", "s")
layout = Layout(modes, dynamics.N, dynamics.dim)
problem = problem_definition.make_problem(layout, LUT_sswi,dynamics)
linearize,values, qp_structure, con_min, con_max, ng = problem

qp_solver = cs.conic(
    "qp_solver",
    "hpipm",
    qp_structure,
    {
        "N": layout.N_ocp, 
        "nx": layout.nx, 
        "nu": layout.nu, 
        "ng": ng,
        "print_time": False, 
        "verbose": False, 
        "error_on_fail": False,
        "hpipm": {
            "mode": "balance",
            "warm_start": 1,
        },
    },
)

In [30]:
ts,tf = LUT_sloc([E0,1])[-1],LUT_floc([E0,1])[-1]
T = float(ts + tf + 0.5*ts)
t_init = cs.vertcat(ts,tf,0.5*ts)
xs_init = initialize_mode(LUT_sloc,dynamics.N["s"],E0) 
xf_init = initialize_mode(LUT_floc,dynamics.N["f"],E0)
x_init = [xs_init, xf_init, initialize_mode(LUT_sloc,dynamics.N["s"],E0,0.5)]
u_init = [np.zeros(dynamics.N[m]-1) for m in modes]
E_init = E0
phi_init = 0.5

p = cs.vertcat(xs_init[:,0],T)

w_min, w_max = problem_definition.make_bounds(layout, x_bounds, u_bounds, t_bounds, E_bounds, phi_bounds)
delta = problem_definition.make_delta(layout)

In [31]:
max_iter = 50
tolerance = 1e-6

history = []
success = False 
w_bar = layout.pack(x_init,u_init,t_init,E_init,phi_init)
x0 = cs.DM.zeros(layout.nw)
start = perf_counter()
for iteration in range(max_iter):
    qp = qp_data(w_bar, p, linearize, con_min, con_max, w_min, w_max, delta)
    qp["x0"] = x0
    result = solve_qp(qp,qp_solver)
    if not qp_solver.stats()["success"]:
        print("Failed ", iteration)
        break
    
    delta_w = result["x"]
    lambda_a = result["lam_a"]
    lambda_x = result["lam_x"]
    mu = max(2000, 1.1*np.max(np.abs(lambda_a)))

    no_improvement, alpha = line_search(w_bar,p,delta_w,values,con_min,con_max,mu)
    w_bar = w_bar + alpha * delta_w 
    _,r = values(w_bar,p)
    max_violation = np.max(np.maximum(con_min-r, 0) + np.maximum(r-con_max, 0))
    success = max_violation < tolerance

    x0 = (1-alpha) * delta_w

    if success:
        print("Success" , iteration)
        break
print(perf_counter() - start)


Success 6
0.08875722099764971
